<a href="https://colab.research.google.com/github/luansalama/Cloud-Blender/blob/main/Improved%20Blender%20Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup
**Make sure to read the instructions carefully!**

If you have other resources used in the Blender project and chose to *make all paths relative*, pack all of them into a zip archive. Alternatively, you can *pack all external file*.

* `blender_version` : Version of blender used to render the scene. You may define your own blender version.
* `blend_file_path` : Path to the blend file after unpacking the zip archive. If blend file is used, this is automatically ignored.
___
* `upload_type` : Select the type of upload method. `gdrive_relative` pulls everything from the folder specified.
* `drive_path` : Path to your blend/zip file relative to the root of your Google Drive if `google_drive` is selected. Must  state the file and its extension (.zip/.blend) **unless** `gdrive_relative` is selected.
* `url_blend` : Specify the URL to the blend/zip file if `url` is selected.
___
* `animation` : Specify whether animation or still image is rendered. If **still image** is used, put the frame number in `start_frame`.
* `start_frame, end_frame` : Specify the start and end frame for animation. You may put same value such as zero for both input to set the default frame in the blend file.
___
* `download_type` : Select the type of download method. `gdrive_direct` enables the frames to be outputted directly to Google Drive (zipping will be disabled).
* `output_name` : Name of the output frames, **do NOT include .blend!** (## for frame number)
* `zip_files` : Archive multiple animation frames automatically into a zip file.
* `drive_output_path` : Path to your frames/zip file in Google Drive.
___
* `gpu_enabled, cpu_enabled` : Toggle GPU and CPU for rendering. CPU might give a slight boost in rendering time but may varies depend on the project.
* `optix_enable` : Enable OptiX which may boost performance, may be incompatible depending on the version of blender, project and GPU allocated

After you are done, go to Runtime > Run All (Ctrl + F9) and upload your files or have Google Drive authorised below. See the [GitHub repo](https://github.com/syn73/blender-colab) for more information.

In [ ]:
# ====== BLENDER RENDER CONFIGURATION ======
blender_version = '4.5.2' #@param ['2.79b', '2.83.20', '2.93.18', '3.3.21', '3.6.23', '4.2.12', '4.5.1', '4.5.2'] {allow-input: true}
blend_file_path = 'path/to/file.blend' #@param {type: 'string'}

#@markdown ---
upload_type = 'google_drive' #@param ['direct', 'google_drive', 'url', 'gdrive_relative'] {allow-input: false}
drive_path = 'colab.blend' #@param {type: 'string'}
url_blend = '' #@param {type: 'string'}

#@markdown ---
start_frame = 1 #@param {type: 'integer'}
end_frame = 100 #@param {type: 'integer'}

#@markdown ---
drive_output_path = 'Output' #@param {type: 'string'}
output_name = 'blender-##' #@param {type: 'string'}

#@markdown ---
gpu_enabled = True #@param {type: 'boolean'}
optix_enabled = True #@param {type: 'boolean'}
cpu_enabled = False #@param {type: 'boolean'}

#@markdown ---
#@markdown ### Multi-Account Distribution
#@markdown Set **TOTAL_WORKERS** to the number of accounts rendering in parallel.
#@markdown Set **WORKER_ID** to 0 on the first account, 1 on the second, etc.
#@markdown Leave both at default to render all frames on a single account.
TOTAL_WORKERS = 1 #@param {type: 'integer'}
WORKER_ID = 0 #@param {type: 'integer'}

#@markdown ---
blender_extra_flags = "" #@param {type: 'string'}
# Advanced: Pass additional Blender flags (e.g., "--verbose")

In [ ]:
# ====== VALIDATE INPUTS ======
print("=" * 60)
print("🔍 VALIDATING CONFIGURATION")
print("=" * 60)

errors = []

# Validate frame range
if start_frame > end_frame:
    errors.append(f"❌ start_frame ({start_frame}) > end_frame ({end_frame})")

# Validate upload settings
if upload_type == 'google_drive' and not drive_path.strip():
    errors.append(f"❌ upload_type is 'google_drive' but drive_path is empty")

if upload_type == 'url' and not url_blend.strip():
    errors.append(f"❌ upload_type is 'url' but url_blend is empty")

# Validate output settings
if not output_name.strip():
    errors.append(f"❌ output_name cannot be empty")

# Show errors if any
if errors:
    print("\n❌ Configuration errors found:\n")
    for error in errors:
        print(f"  {error}")
    print("\n⚠️  Please fix the errors above and re-run this cell.\n")
    raise SystemExit("Fix configuration errors above")

print("✅ Configuration validated - all settings OK")
print("=" * 60)

In [ ]:
# ====== CHECK GPU & SYSTEM ======
import os
import subprocess
from pathlib import Path
import time

print("\n" + "=" * 60)
print("🔧 SYSTEM SETUP")
print("=" * 60)

os.chdir('/content')
print(f"Working directory: {Path.cwd()}")

# Check GPU availability
print("\n📡 Checking GPU...")
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=gpu_name', '--format=csv,noheader'],
        capture_output=True,
        text=True,
        timeout=10
    )
    if result.returncode == 0:
        gpu_name = result.stdout.strip()
        print(f"✅ GPU available: {gpu_name}")

        # Check for unsupported GPUs
        if "Tesla K80" in gpu_name and optix_enabled:
            print("⚠️  K80 GPU detected - OptiX not supported, using CUDA")
            optix_enabled = False
    else:
        print("⚠️  GPU not available - will use CPU (slow!)")
        gpu_enabled = False
except Exception as e:
    print(f"⚠️  Could not detect GPU: {e}")
    gpu_enabled = False

# Setup tcmalloc for memory optimization
print("\n💾 Setting up memory allocator...")
os.environ["LD_PRELOAD"] = ""

try:
    subprocess.run(['apt', 'remove', '-y', 'libtcmalloc-minimal4'],
                   capture_output=True, timeout=30)
    subprocess.run(['apt', 'install', '-y', 'libtcmalloc-minimal4'],
                   capture_output=True, timeout=30)

    tcmalloc_path = "/usr/lib/x86_64-linux-gnu/libtcmalloc_minimal.so.4.5.9"
    if Path(tcmalloc_path).exists():
        os.environ["LD_PRELOAD"] = tcmalloc_path
        print(f"✅ tcmalloc enabled: {tcmalloc_path}")
    else:
        print(f"⚠️  tcmalloc not found at {tcmalloc_path}")
except Exception as e:
    print(f"⚠️  Could not setup tcmalloc: {e}")

print("=" * 60)

In [ ]:
# ====== MOUNT DRIVE & UPLOAD FILES ======
import shutil
from google.colab import files, drive

print("\n" + "=" * 60)
print("📥 LOADING BLEND FILE")
print("=" * 60)

uploaded_filename = ""

# Check if we need Google Drive
if (upload_type in ['google_drive', 'gdrive_relative'] or
    download_type in ['google_drive', 'gdrive_direct']):
    print("\n📁 Mounting Google Drive...")
    drive.mount('/drive')
    print("✅ Google Drive mounted at /drive")

# Load blend file based on upload type
if upload_type == 'direct':
    print("\n📤 Uploading file directly...")
    uploaded = files.upload()
    for fn in uploaded.keys():
        uploaded_filename = fn
    print(f"✅ Uploaded: {uploaded_filename}")

elif upload_type == 'url':
    print(f"\n🌐 Downloading from URL: {url_blend}")
    result = subprocess.run(['wget', '-nc', url_blend],
                          capture_output=True, timeout=300)
    if result.returncode == 0:
        uploaded_filename = os.path.basename(url_blend)
        print(f"✅ Downloaded: {uploaded_filename}")
    else:
        print(f"❌ Download failed: {result.stderr.decode()}")
        raise SystemExit("Failed to download blend file")

elif upload_type == 'google_drive':
    print(f"\n📂 Loading from Google Drive: {drive_path}")
    try:
        shutil.copy(f'/drive/MyDrive/{drive_path}', '.')
        uploaded_filename = os.path.basename(drive_path)
        print(f"✅ Loaded: {uploaded_filename}")
    except FileNotFoundError:
        print(f"❌ File not found in Google Drive: {drive_path}")
        raise SystemExit("File not found in Google Drive")

print("=" * 60)

In [ ]:
# ====== PREPARE DIRECTORIES ======
from pathlib import Path

print("\n" + "=" * 60)
print("🗂️  PREPARING DIRECTORIES")
print("=" * 60)

CONTENT = Path('/content')
render_dir = CONTENT / 'render'
output_dir = CONTENT / 'output'

# Clean and create directories
if render_dir.exists():
    print(f"\n🗑️  Cleaning: {render_dir}")
    shutil.rmtree(render_dir)

render_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ Created render directory: {render_dir}")
print(f"✅ Created output directory: {output_dir}")

# Handle different upload types
if upload_type == 'gdrive_relative':
    print(f"\n📋 Copying folder: {drive_path}")
    drive_path_fixed = drive_path if drive_path.endswith('/') else drive_path + '/'
    subprocess.run(['cp', '-r', f'/drive/MyDrive/{drive_path_fixed}.', str(render_dir)],
                  check=True)
    print(f"✅ Copied files to {render_dir}")

elif uploaded_filename.lower().endswith('.zip'):
    print(f"\n📦 Extracting: {uploaded_filename}")
    subprocess.run(['unzip', '-o', uploaded_filename, '-d', str(render_dir)],
                  check=True, capture_output=True)
    print(f"✅ Extracted to {render_dir}")

elif uploaded_filename.lower().endswith('.blend'):
    print(f"\n📝 Copying blend file: {uploaded_filename}")
    shutil.copy(uploaded_filename, render_dir)
    blend_file_path = uploaded_filename
    print(f"✅ Copied to {render_dir}")

elif uploaded_filename:
    ext = Path(uploaded_filename).suffix.lower()
    print(f"\n❌ Invalid file format: {ext}")
    print(f"   Valid formats: .blend, .zip")
    raise SystemExit("Invalid file extension")

print("=" * 60)

In [ ]:
# ====== DOWNLOAD BLENDER ======
import requests
import sys

print("\n" + "=" * 60)
print("⬇️  DOWNLOADING BLENDER")
print("=" * 60)

# Blender version URLs
blender_url_dict = {
    '2.79b'   : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender2.79/blender-2.79b-linux-glibc219-x86_64.tar.bz2",
    '2.83.20' : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender2.83/blender-2.83.20-linux-x64.tar.xz",
    '2.93.18' : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender2.93/blender-2.93.18-linux-x64.tar.xz",
    '3.3.21'  : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender3.3/blender-3.3.21-linux-x64.tar.xz",
    '3.6.23'  : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender3.6/blender-3.6.23-linux-x64.tar.xz",
    '4.2.12'  : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender4.2/blender-4.2.12-linux-x64.tar.xz",
    '4.5.1'   : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender4.5/blender-4.5.1-linux-x64.tar.xz",
    '4.5.2'   : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender4.5/blender-4.5.2-linux-x64.tar.xz",
}

# Get URL for requested version
if blender_version in blender_url_dict:
    blender_url = blender_url_dict[blender_version]
    print(f"✓ Found predefined URL for version {blender_version}")
else:
    # Auto-construct URL for custom versions
    major_minor = ".".join(blender_version.split('.')[:2])
    blender_url = f"https://ftp.nluug.nl/pub/graphics/blender/release/Blender{major_minor}/blender-{blender_version}-linux-x64.tar.xz"
    print(f"✓ Constructing URL for version {blender_version}")

# Verify URL exists
print(f"\n🔗 Checking URL accessibility...")
try:
    response = requests.head(blender_url, allow_redirects=True, timeout=10)
    if response.status_code == 200:
        base_url = os.path.basename(blender_url)
        file_size = int(response.headers.get('content-length', 0))
        file_size_mb = file_size / (1024 * 1024)
        print(f"✅ URL accessible")
        print(f"   File: {base_url}")
        print(f"   Size: {file_size_mb:.1f} MB")
    else:
        print(f"❌ Error checking URL (status {response.status_code})")
        print(f"   You may need to define the URL manually above")
        raise SystemExit("Cannot access Blender download URL")
except Exception as e:
    print(f"❌ Error checking URL: {e}")
    raise SystemExit("Cannot access Blender download URL")

# Download Blender with progress
print(f"\n📥 Downloading Blender {blender_version}...")
print(f"   This may take several minutes depending on file size...")

blender_dir = CONTENT / blender_version
blender_dir.mkdir(exist_ok=True, parents=True)

def download_with_progress(url, filename, chunk_size=8192):
    """Download file with progress bar"""
    response = requests.get(url, stream=True, timeout=300)
    total_size = int(response.headers.get('content-length', 0))

    downloaded = 0
    with open(filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

                # Print progress every 10MB
                if downloaded % (10 * 1024 * 1024) == 0 or downloaded == total_size:
                    percent = (downloaded / total_size * 100) if total_size > 0 else 0
                    mb_downloaded = downloaded / (1024 * 1024)
                    mb_total = total_size / (1024 * 1024)

                    # Progress bar
                    bar_length = 30
                    filled = int(bar_length * downloaded / total_size) if total_size > 0 else 0
                    bar = '█' * filled + '░' * (bar_length - filled)

                    print(f"\r   [{bar}] {percent:.1f}% ({mb_downloaded:.1f}/{mb_total:.1f} MB)", end='', flush=True)

    print()  # New line after progress
    return filename

# Download
download_with_progress(blender_url, base_url)
print(f"✅ Download complete: {base_url}")

# Extract Blender with progress
print(f"\n📦 Extracting Blender to {blender_dir}...")
print(f"   This may take a couple minutes...")

# Show tar progress
result = subprocess.run([
    'tar', '-xvf', base_url,
    '-C', str(blender_dir),
    '--strip-components=1'
], capture_output=True, text=True)

# Count extracted files
extracted_lines = result.stdout.count('\n')
print(f"✅ Extraction complete: {extracted_lines} files extracted")
print(f"✅ Blender ready at: {blender_dir / 'blender'}")

# Verify Blender executable exists
blender_exe = blender_dir / 'blender'
if blender_exe.exists():
    print(f"✅ Verified: Blender executable found")
    # Get file size
    exe_size = blender_exe.stat().st_size / (1024 * 1024)
    print(f"   Size: {exe_size:.1f} MB")
else:
    print(f"❌ Error: Blender executable not found!")
    raise SystemExit("Blender installation failed")

print("=" * 60)

In [ ]:
# ====== GENERATE GPU CONFIGURATION SCRIPT ======

print("\n" + "=" * 60)
print("⚙️  GENERATING GPU CONFIGURATION")
print("=" * 60)

# Create the Python script for Blender GPU setup
gpu_script = f"""import re
import bpy

scene = bpy.context.scene
scene.cycles.device = 'GPU'
prefs = bpy.context.preferences
prefs.addons['cycles'].preferences.get_devices()
cprefs = prefs.addons['cycles'].preferences

print(cprefs)

# Try to set compute device type
for compute_device_type in ('CUDA', 'OPENCL', 'NONE'):
    try:
        cprefs.compute_device_type = compute_device_type
        print(f'Device type set to: {{compute_device_type}}')
        break
    except TypeError:
        pass

# Configure individual devices
for device in cprefs.devices:
    device_name = device.name.lower()

    # Enable GPU devices (except Intel)
    if 'intel' not in device_name:
        print(f'Enabling GPU: {{device.name}}')
        device.use = {gpu_enabled}
    else:
        # Intel devices use CPU
        print(f'Setting Intel to CPU: {{device.name}}')
        device.use = {cpu_enabled}

print("GPU configuration complete")
"""

script_path = CONTENT / 'setgpu.py'
with open(script_path, 'w') as f:
    f.write(gpu_script)

print(f"✅ GPU script created: {script_path}")

# Determine which renderer to use
if optix_enabled:
    renderer = "OPTIX"
    print(f"⚡ Using renderer: OPTIX (OptiX acceleration)")
    print(f"   Note: If OptiX fails, Blender will fall back to CUDA")
else:
    renderer = "CUDA"
    print(f"⚡ Using renderer: CUDA")

print("=" * 60)

In [ ]:
# ======================================================================
# HYBRID RENDER & SYNC PIPELINE
# ======================================================================

import os, subprocess, shutil, time, numpy as np
from pathlib import Path
from google.colab import runtime
from IPython.display import Audio, display

# ======================================================================
# SECTION 1 — MULTI-ACCOUNT DISTRIBUTION
# Set these differently for each Colab account running in parallel.
# TOTAL_WORKERS = 1 disables distribution (single account, all frames).
# ======================================================================

TOTAL_WORKERS = 1   # Total accounts rendering in parallel
WORKER_ID     = 0   # This account: Account 1 = 0, Account 2 = 1, etc.

# ======================================================================
# SECTION 2 — RESOLVE PATHS FROM NOTEBOOK CONTEXT
# These variables are already set by earlier notebook cells —
# we just resolve and validate them here.
# ======================================================================

CONTENT         = Path('/content')
blender_exe     = CONTENT / blender_version / 'blender'
blend_file      = CONTENT / 'render' / blend_file_path

# Notebook outputs frames to /content/output/
local_frame_dir = CONTENT / 'output'
local_frame_dir.mkdir(parents=True, exist_ok=True)

# Drive destination — matches drive_output_path from config cell
drive_output_dir = Path(f'/drive/MyDrive/{drive_output_path}')
drive_output_dir.mkdir(parents=True, exist_ok=True)

# ======================================================================
# SECTION 3 — HELPERS
# ======================================================================

def play_pin_pon_pun():
    sr  = 44100
    dur = 0.3
    gen_tone = lambda f, d: (
        np.sin(2 * np.pi * f * np.linspace(0, d, int(sr * d)))
        * np.exp(-3 * np.linspace(0, d, int(sr * d)) / d)
    )
    full_signal = np.concatenate([
        gen_tone(880, dur), np.zeros(int(sr * 0.05)),
        gen_tone(659, dur), np.zeros(int(sr * 0.05)),
        gen_tone(523, dur)
    ])
    display(Audio(full_signal, rate=sr, autoplay=True))

def frame_name_str(frame):
    """
    Replicates Blender's ## padding logic from the notebook's output_name.
    e.g. output_name='blender-##', frame=5  → 'blender-0005'
         output_name='frame_####', frame=42 → 'frame_0042'
    """
    padding = output_name.count('#')
    base    = output_name.replace('#' * padding, '')
    return f"{base}{frame:0{padding}d}"

# ======================================================================
# SECTION 4 — PREFLIGHT CHECKS
# ======================================================================

print("=" * 60)
print("🔍 PREFLIGHT CHECKS")
print("=" * 60)

preflight_ok = True

if not blender_exe.exists():
    print(f"❌ Blender executable not found: {blender_exe}")
    preflight_ok = False
else:
    print(f"✅ Blender       : {blender_exe}")

if not blend_file.exists():
    print(f"❌ Blend file not found: {blend_file}")
    preflight_ok = False
else:
    blend_size = blend_file.stat().st_size / (1024 * 1024)
    print(f"✅ Blend file    : {blend_file.name} ({blend_size:.1f} MB)")

setgpu_path = CONTENT / 'setgpu.py'
if not setgpu_path.exists():
    print(f"❌ setgpu.py not found — run the GPU Configuration cell first")
    preflight_ok = False
else:
    print(f"✅ GPU script    : {setgpu_path}")

if not drive_output_dir.exists():
    print(f"❌ Drive output folder could not be created: {drive_output_dir}")
    preflight_ok = False
else:
    print(f"✅ Drive output  : {drive_output_dir}")

if not preflight_ok:
    raise SystemExit("❌ Preflight failed — fix the errors above and re-run.")

print("=" * 60)

# ======================================================================
# SECTION 5 — FRAME DISTRIBUTION
# ======================================================================

all_frames = list(range(start_frame, end_frame + 1)) if animation else [start_frame]

# Distribute frames across workers via modulo (interleaved, no overlap)
frames       = [f for i, f in enumerate(all_frames) if i % TOTAL_WORKERS == WORKER_ID]
total_frames = len(frames)

# ======================================================================
# SECTION 6 — STARTUP HEADER
# ======================================================================

print(f"\n{'='*60}")
print(f"🚀 HYBRID RENDER & SYNC PIPELINE")
print(f"{'='*60}")
print(f"📋 Scene        : {blend_file.name}")
print(f"🖥️  Blender      : {blender_exe.parent.name}")
print(f"⚡ Renderer     : {renderer}")
print(f"🎞️  All frames   : {start_frame} → {end_frame}  ({len(all_frames)} total)")
print(f"📁 Local output : {local_frame_dir}")
print(f"☁️  Drive output : {drive_output_dir}")

if TOTAL_WORKERS > 1:
    print(f"{'='*60}")
    print(f"👥 DISTRIBUTED MODE: Worker {WORKER_ID + 1} of {TOTAL_WORKERS}")
    print(f"🎯 This worker handles {total_frames} frames")
    print(f"   First few: {frames[:5]}{'...' if len(frames) > 5 else ''}")
else:
    print(f"🎯 Rendering all {total_frames} frames (single worker)")

print(f"{'='*60}\n")

# ======================================================================
# SECTION 7 — RENDER LOOP
# ======================================================================

failed_frames         = []
skipped_frames        = []
rendered_this_session = 0
render_start_time     = time.time()

# Read Drive directory ONCE before the loop into a local set —
# avoids one stat/glob call to Drive per frame (critical at scale).
# The cache is updated after each successful upload to stay accurate.
print(f"📂 Reading Drive directory...", end=" ", flush=True)
try:
    drive_files_cache = set(os.listdir(drive_output_dir))
    print(f"✅ {len(drive_files_cache)} file(s) already on Drive.\n")
except Exception as e:
    print(f"⚠️  Could not read Drive directory: {e}")
    print(f"   Auto-resume disabled — all frames will be rendered.\n")
    drive_files_cache = set()

for idx, frame in enumerate(frames, start=1):
    fname        = frame_name_str(frame)
    local_output = local_frame_dir / fname   # Blender appends extension itself

    # ── AUTO-RESUME: check against in-memory cache, not a fresh Drive call ──
    if any(fn.startswith(fname) for fn in drive_files_cache):
        print(f"⏩ [{idx}/{total_frames}] Frame {frame} already on Drive — skipping.")
        skipped_frames.append(frame)
        continue

    # ── RENDER (single frame via -f flag) ──────────────────────────────
    cmd = [
        str(blender_exe),
        "-b",       str(blend_file),
        "-P",       str(setgpu_path),
        "-E",       "CYCLES",
        "-o",       str(local_output),
        "-noaudio",
        "-f",       str(frame),
        "--",
        "--cycles-device", renderer,
    ]

    if blender_extra_flags.strip():
        cmd.extend(blender_extra_flags.strip().split())

    frame_start = time.time()
    print(f"🎬 [{idx}/{total_frames}] Rendering frame {frame}...", end=" ", flush=True)

    result  = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - frame_start

    # ── VALIDATE OUTPUT ────────────────────────────────────────────────
    matches = list(local_frame_dir.glob(f"{fname}*"))
    if not matches:
        print(f"❌ FAILED ({elapsed:.1f}s)")
        if result.stderr:
            print(f"   ⚠️  Blender error: {result.stderr[-300:].strip()}")
        else:
            print(f"   ⚠️  No error output captured — check .blend path and GPU config.")
        failed_frames.append(frame)
        continue

    rendered_file = matches[0]
    file_size_mb  = rendered_file.stat().st_size / (1024 * 1024)
    rendered_this_session += 1
    print(f"✅ Done ({elapsed:.1f}s | {file_size_mb:.2f} MB)")

    if file_size_mb < 0.01:
        print(f"   ⚠️  Warning: Frame {frame} is unusually small ({file_size_mb:.3f} MB) — may be corrupt or blank.")

    # ── UPLOAD TO DRIVE (immediately after render) ─────────────────────
    print(f"   ☁️  Uploading to Drive...", end=" ", flush=True)
    try:
        dest = drive_output_dir / rendered_file.name
        shutil.copy(str(rendered_file), str(dest))
        drive_files_cache.add(rendered_file.name)   # keep cache in sync
        print(f"✅ {dest.name}")
    except Exception as e:
        print(f"❌ Upload failed: {e}")
        print(f"   ⚠️  Frame {frame} is safe locally at {rendered_file} but NOT on Drive.")

    # ── PROGRESS & ETA every 10 frames ────────────────────────────────
    if rendered_this_session > 0 and (idx % 10 == 0 or idx == total_frames):
        total_elapsed    = time.time() - render_start_time
        avg_per_frame    = total_elapsed / rendered_this_session
        frames_remaining = total_frames - idx
        eta              = avg_per_frame * max(frames_remaining, 0)
        print(f"\n   📊 Progress  : {idx}/{total_frames} frames")
        print(f"   ⏩ Skipped   : {len(skipped_frames)} (already on Drive)")
        print(f"   ✅ Rendered  : {rendered_this_session} this session")
        print(f"   ⏱️  Elapsed   : {total_elapsed / 60:.1f} min")
        print(f"   🏁 ETA       : {eta / 60:.1f} min\n")

# ======================================================================
# SECTION 8 — FINAL SUMMARY
# ======================================================================

total_time = time.time() - render_start_time

print(f"\n{'='*60}")
print(f"📊 RENDER SUMMARY — Worker {WORKER_ID + 1} of {TOTAL_WORKERS}")
print(f"{'='*60}")
print(f"✅ Rendered this session : {rendered_this_session} frames")
print(f"⏩ Skipped (on Drive)    : {len(skipped_frames)} frames")

if failed_frames:
    print(f"❌ Failed                : {len(failed_frames)} frames → {failed_frames}")
    print(f"   ⚠️  Re-run this cell to retry — auto-resume will skip completed frames.")
else:
    print(f"🎉 No failures!")

# Global progress check when using multiple workers —
# does ONE fresh os.listdir() at the end for an accurate cross-worker count
if TOTAL_WORKERS > 1:
    print(f"\n{'='*60}")
    print(f"🌐 GLOBAL PROGRESS (all workers combined)")
    print(f"{'='*60}")
    try:
        final_drive_files = set(os.listdir(drive_output_dir))
        all_on_drive      = [f for f in all_frames
                             if any(fn.startswith(frame_name_str(f)) for fn in final_drive_files)]
        still_missing     = [f for f in all_frames if f not in all_on_drive]

        print(f"☁️  On Drive  : {len(all_on_drive)}/{len(all_frames)} frames")
        if still_missing:
            print(f"⏳ Missing    : {len(still_missing)} frames → "
                  f"{still_missing[:10]}{'...' if len(still_missing) > 10 else ''}")
        else:
            print(f"🎉 ALL {len(all_frames)} FRAMES ARE ON DRIVE — render complete!")
    except Exception as e:
        print(f"⚠️  Could not read Drive directory for global check: {e}")

print(f"\n⏱️  Session time : {total_time / 60:.1f} minutes")
print(f"{'='*60}")

# ======================================================================
# SECTION 9 — SHUTDOWN
# ======================================================================

print(f"\n🏁 PIPELINE COMPLETE. TERMINATING RUNTIME IN 10s...")
play_pin_pon_pun()
for i in range(10, 0, -1):
    print(f"\r⏱️  Closing in {i}s", end="", flush=True)
    time.sleep(1)
runtime.unassign()

In [ ]:
# ====== AUTO-SHUTDOWN GATE WITH ALERT ======
import time
import numpy as np
from google.colab import runtime
from IPython.display import Audio, display

def play_pin_pon_pun():
    sr = 44100  # Sample rate
    dur = 0.3   # Duration of each tone in seconds

    def gen_tone(freq, duration):
        t = np.linspace(0, duration, int(sr * duration))
        # Applied a small fade-out (envelope) to prevent clicking
        envelope = np.exp(-3 * t / duration)
        return np.sin(2 * np.pi * freq * t) * envelope

    # Frequencies: A5 (880Hz), E5 (659Hz), C5 (523Hz)
    pin = gen_tone(880, dur)
    pon = gen_tone(659, dur)
    pun = gen_tone(523, dur)

    # Combine tones with a tiny bit of silence between them
    silence = np.zeros(int(sr * 0.05))
    full_signal = np.concatenate([pin, silence, pon, silence, pun])

    display(Audio(full_signal, rate=sr, autoplay=True))

print("\n" + "=" * 60)
print("🏁 RENDER COMPLETE - EXPORTING DATA")
print("=" * 60)

# Play the sequence
play_pin_pon_pun()

# 10-second countdown for Google Drive sync
for i in range(60, 0, -1):
    print(f"\r⏱️  Syncing to Drive... Shutting down in {i}s (Stop cell to cancel)", end="", flush=True)
    time.sleep(1)

print("\n\n🔌 Runtime unassigned. Compute units saved.")
runtime.unassign()

## Disclaimer
Google Colab is targeted to researchers and students to run AI/ML tasks, data analysis and education, not rendering 3D scenes. Because the computing power provided are free, the usage limits, idle timeouts and speed of the rendering may varies time by time. [Colab Pro and Colab Pro+](https://colab.research.google.com/signup) are available for those who wanted to have more powerful GPU and longer runtimes for rendering. See the [FAQ](https://research.google.com/colaboratory/faq.html) for more info. In some cases, it might be faster to use an online Blender renderfarm.

## License
```
MIT License

Copyright (c) 2020-2022 ynshung

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```